# Coachly NLU - Colab Drive Notebook (Qwen 0.5B)
Pipeline: mount drive -> setup files -> build dataset -> train QLoRA -> noisy STT eval.


In [ ]:
# 1) Mount Drive + repo path
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/voice-ml-recognizer'
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Manca {REPO_DIR}.')
%cd {REPO_DIR}


In [ ]:
# 2) GPU + deps
!nvidia-smi
!python -V

# Dipendenze minime stabili
!pip install -q -U "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" sentencepiece protobuf huggingface_hub


In [ ]:
# 3) Ensure scripts in repo root (copy from refactor if missing)
import os, shutil

pairs = [
    ("dataset_creator_v2.py",            "refactor/dataset_creator_v2.py"),
    ("augment.py",                        "refactor/augment.py"),
    ("colab_functiongemma_train.py",       "refactor/colab_functiongemma_train.py"),
]
for dst, src in pairs:
    if not os.path.exists(dst):
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"Copied {src} -> {dst}")
        else:
            raise FileNotFoundError(f"Mancano sia {dst} che {src}")
    else:
        print(f"OK: {dst}")


In [ ]:
# 4) Compatibility patch for TrainingArguments (eval_strategy vs evaluation_strategy)
from pathlib import Path
import re

p = Path('colab_functiongemma_train.py')
s = p.read_text(encoding='utf-8')

# Rendi il file compatibile con le versioni che usano eval_strategy
if 'evaluation_strategy=' in s and 'inspect.signature(TrainingArguments.__init__)' not in s:
    s = s.replace('evaluation_strategy=', 'eval_strategy=')

p.write_text(s, encoding='utf-8')
print('Patched compatibility in', p)


In [ ]:
# 5) Build dataset v2 (80k samples, pattern-based, STT names in label)
TOTAL = 80000  # aumenta a 100000 se hai tempo su A100

!python dataset_creator_v2.py --total {TOTAL} --output_dir data_v2 --md_dir refactor/exercises

# Augmenta il train set (+50% varianti surface)
!python augment.py --input data_v2/train.jsonl --output data_v2/train_aug.jsonl --factor 1.5

import json
from pathlib import Path
m = json.loads(Path("data_v2/metadata.json").read_text(encoding="utf-8"))
print("Sizes:", m["sizes"])
print("Actions all:", m["stats"]["all"]["action"])
print("Exercise pool:", m["exercise_pool_size"], "exercises,", m["total_aliases"], "aliases")


In [ ]:
# 6) Train QLoRA on Qwen 0.5B
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "output/functiongemma_qlora_v2"

# usa train_aug.jsonl (originale + augmented) come train set
cmd = f"python colab_functiongemma_train.py --data_dir data_v2 --train_file train_aug.jsonl --output_dir {OUTPUT_DIR} --base_model {BASE_MODEL}"
print(cmd)
!


In [ ]:
# 7) Quick metrics (generated by train script)
import json
from pathlib import Path
q = Path('output/functiongemma_qlora/quick_eval.json')
if q.exists():
    print(json.dumps(json.loads(q.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('quick_eval.json non trovato (training non concluso)')


In [ ]:
# 8) Noisy STT evaluation
import json
import os
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_DIR = 'output/functiongemma_qlora/adapter'
META_PATH = 'data/metadata.json'

if not os.path.isdir(ADAPTER_DIR):
    raise FileNotFoundError(f'Adapter non trovato: {ADAPTER_DIR}. Prima completa il training.')

system_prompt = json.loads(open(META_PATH, encoding='utf-8').read())['system_prompt'] if os.path.exists(META_PATH) else 'You are Coachly NLU. Return strict JSON only.'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()


def extract_json(s: str):
    s = s.strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    m = re.search(r'\{.*\}', s, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def predict(text: str):
    msgs = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': text},
    ]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inp,
            max_new_tokens=180,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    txt = tokenizer.decode(out[0][inp.shape[-1]:], skip_special_tokens=True).strip()
    parsed = extract_json(txt)
    return txt, parsed


noisy_cases = [
    ('aggiungii bencc press 3x10 a cedimnto', 'ADD_EXERCISE'),
    ('fatto deadlif 5 rep 140 kg', 'LOG_SET'),
    ('rimouvi lat mascin', 'DELETE_EXERCISE'),
    ('no aspetta aggiungi panca piana 3x8 e trazioni 4x6', 'ADD_EXERCISE'),
    ('uhm metti squat 4 serie da 8 e push ap 3x15 drop set', 'ADD_EXERCISE'),
    ('correggo togli pushup e rematore', 'DELETE_EXERCISE'),
    ('modifca deadlift a 4x6 120kg', 'UPDATE_SET'),
    ('aggiorna trazioni 5x5 con pausa', 'UPDATE_SET'),
    ('ho fatto bench press 8 rep 80 kilo to failur', 'LOG_SET'),
    ('done benh press 3 set of 8 rep 80 kg', 'LOG_SET'),
    ('add squat 5x5 100kg and pull up 4x6', 'ADD_EXERCISE'),
    ('wait no actually remove lath pull down', 'DELETE_EXERCISE'),
    ('change deadlift to 3x5 at 140 kg', 'UPDATE_SET'),
    ('how many calories i burn today', 'UNKNOWN'),
    ('fammi una scheda petto tricipiti pls', 'UNKNOWN'),
]

ok, valid = 0, 0
for i, (text, expected) in enumerate(noisy_cases, start=1):
    raw, parsed = predict(text)
    pred_action = parsed.get('action') if isinstance(parsed, dict) else None
    is_valid = isinstance(parsed, dict)
    is_ok = (pred_action == expected)
    valid += int(is_valid)
    ok += int(is_ok)
    print(f'[{i:02d}] expected={expected:16s} pred={str(pred_action):16s} valid_json={is_valid} ok={is_ok}')
    print('  text:', text)
    print('  out :', json.dumps(parsed, ensure_ascii=False) if is_valid else raw)
    print()

print('---')
print(f'Action accuracy: {ok}/{len(noisy_cases)} = {ok/len(noisy_cases):.1%}')
print(f'Valid JSON rate: {valid}/{len(noisy_cases)} = {valid/len(noisy_cases):.1%}')
